# Getting started with biogate

biogate validates biological identifiers and inputs. It answers one
question, "is this a valid identifier?", with the same verdict in Python
and R. This notebook walks through the sources, the checking modes, and the
validation adapters using only offline data, so every cell here runs without
a network.

In [ ]:
import biogate as bg

bg.sources()

## Pattern mode: is the string well-formed?

`pattern` mode is offline and deterministic. It checks the shape of an
identifier against the source's pattern. It does not check that the
identifier exists.

`check_id()` returns a list of `Result` records, one per input, in the same
order and length as the input. An invalid value is never collapsed into a
bare `False`: the record tells you why.

In [ ]:
results = bg.check_id(["MONDO:0005148", "mondo:5148", "GO:0006915"], source_db="mondo")
for r in results:
    print(r.input, r.valid, r.normalized, r.suggestion)

The `normalized` field holds the canonical form of a valid input. The
`suggestion` field holds a best-effort correction for an invalid but
mappable input, such as the lowercase prefix above.

For just the verdict, use `is_valid_id()`.

In [ ]:
bg.is_valid_id(["P04637", "p04637"], source_db="uniprot")

## Cache mode: does the identifier exist in a snapshot?

`cache` mode checks existence against a pinned, offline snapshot. A small
`sample` snapshot ships with the package, so these cells need no downloads.
A real analysis pins a dated snapshot instead.

In [ ]:
for r in bg.check_id(
    ["MONDO:0005148", "MONDO:9999999"], source_db="mondo", how="cache", version="sample"
):
    print(r.input, r.valid)

`MONDO:9999999` is well-formed, so it passes `pattern` mode, but it is not in
the snapshot, so it fails `cache` mode. Choosing the mode is choosing how
strict you want the check to be.

## Remote mode

`remote` mode checks existence live against a source API, so it needs a
network and is not run here. The call looks like this:

```python
# Live check against the Ensembl REST API.
bg.check_id("ENSG00000139618", source_db="ensembl", how="remote")

# existence mode uses a snapshot when available, otherwise falls back to remote.
bg.check_id("MONDO:0005148", source_db="mondo", how="existence")
```

A network failure in `remote` mode raises an error. It never returns a silent
`False`, so a failed lookup cannot be mistaken for an absent identifier.

## Species and version awareness

Some sources are species-aware. For Ensembl the species is encoded in the id,
so `pattern` mode can reject a well-formed id that belongs to the wrong
species. `species` accepts a name or an NCBI taxon id such as `9606`.

In [ ]:
# ENSMUSG is a mouse gene id.
print(bg.is_valid_id("ENSMUSG00000059552", source_db="ensembl", species="mus_musculus"))
print(bg.is_valid_id("ENSMUSG00000059552", source_db="ensembl", species="homo_sapiens"))

## HGVS variant syntax

The `hgvs` source checks the syntax of an HGVS sequence variant name. This is
a grammar check in `pattern` mode. It confirms the shape of the variant. It
does not check coordinates or that the variant exists.

In [ ]:
bg.is_valid_id(
    [
        "NM_004006.2:c.4375C>T",
        "NP_003997.1:p.(Gly56Ala)",
        "NM_004006.2:c.76insG",
    ],
    source_db="hgvs",
)

The last one is invalid: an insertion must sit between two flanking
positions, so it needs a range such as `c.76_77insG`.

## Validation framework adapters

The adapters wrap the core classifier so it plugs into common validation
frameworks. They never reimplement any checks. Install them with
`pip install biogate[adapters]`.

### pydantic

`Id()` returns a validating string type for a model field.

In [ ]:
from pydantic import BaseModel

from biogate.types import Id

MondoId = Id("mondo")


class Row(BaseModel):
    term: MondoId


print(Row(term="MONDO:0005148").term)

### pandera

`is_id()` returns a pandera `Check` for a column of identifiers.

In [ ]:
import pandas as pd
import pandera.pandas as pa

from biogate.checks import is_id

schema = pa.DataFrameSchema({"term": pa.Column(str, is_id("mondo"))})
df = pd.DataFrame({"term": ["MONDO:0005148", "MONDO:0018076"]})
schema.validate(df)

## Summary

- `pattern` mode checks shape, `cache` and `remote` check existence, and
  `existence` uses a snapshot first and remote as a fallback.
- Results are vectorized and preserve input order and length. Errors are
  explicit.
- The same inputs give the same verdicts in the R package, which is enforced
  by a shared conformance corpus.